# JWST Multiphase Data Reduction: Vector Stage Analytics & Diagnostics
**Project Context:** Young Substellar Horizons Core Sample Analysis (IC 348 / HH 211)  
**Author:** Oleg G.Kapranov  
**Pipeline Release:** v4.2.0 (SIMD Production Batch Performance Logging Engine)  

---

### 1. Environment Matrix Loading & Initialization
We initialize our Python scientific stack and register the automated vectorized reduction engine components.

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from run_batch_substellar_reduction import run_production_pipeline_batch
import warnings
from astropy.wcs import FITSFixedWarning
warnings.simplefilter('ignore', category=FITSFixedWarning)

print("✅ FITS formatting warnings silenced for clean diagnostic execution log output!")

# Synchronize your local file hooks to match your active repository layout exactly
stage3_dir = "./processed_stage3/"

# Point to your actual 1.4 GB Stage 3 FITS science image mosaic file
mosaic_path = os.path.join(stage3_dir, "hh211_stacked_mosaic_f150w2_f162m_i2d.fits")

# Verify that the input file matching your candidate listing is registered
catalog_in = os.path.join(stage3_dir, "ic348_brown_dwarf_candidates_final.csv")
catalog_out = os.path.join(stage3_dir, "ic348_combined_master_science_catalog.csv")

print(f"🛰️ Target Mosaic Linked: {mosaic_path}")
print(f"📂 Verification Catalog : {catalog_in}")

### 2. High-Performance Execution & Profiling
We run the batch data reduction runtime pipeline and time its performance to track the latency scaling across the array-wide SIMD loops.

In [ ]:
print("🚀 Booting Production Vector Optimization Run...\n")
start_time = time.perf_counter()

run_production_pipeline_batch(mosaic_path, catalog_in, catalog_out)

end_time = time.perf_counter()
execution_duration = end_time - start_time

print(f"\n⏱️ TIMER SUMMARY:")
print(f"   --> Total Processing Latency for 205 Targets: {execution_duration:.4f} seconds")
print(f"   --> Average Computational Latency per Target: {(execution_duration / 205)*1000:.2f} milliseconds")

### 3. Structural Data Quality & Convergence Verification
We load the newly exported master database catalog file to audit the output properties and ensure the self-healing clipping loops have stabilized without dropping target metrics.

In [ ]:
# Load generated results matrix
results_df = pd.read_csv(catalog_out)

print("=== PIPELINE INTEGRITY METRICS ===")
print(f"• Total successfully processed rows : {len(results_df)} / 205")
print(f"• Missing / NaN coordinate artifacts : {results_df['AB_Magnitude'].isna().sum()}")
print(f"• Sky RMS Noise Floor Baseline (Mean): {results_df['Sky_RMS_Noise'].mean():.6f} DN")
print(f"• Aperture Flux Extreme Saturation  : {results_df['Net_Aperture_Flux'].max():.2f} DN")

assert len(results_df) == 205, "⚠️ Integrity Warning: Discrepancy found inside target array footprint row counts!"
print("\n🎉 SUCCESS: Data structures verified completely clean.")

### 4. Interactive Statistical Distribution Visualizations
We construct diagnostic distribution plots to verify our sigma-clipped sky variance floor and visually audit our computed AB Magnitudes sequence.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- EMERGENCY WORKSPACE DATAFRAME RE-LOAD ---
stage3_dir = "./processed_stage3/"
catalog_out = os.path.join(stage3_dir, "ic348_combined_master_science_catalog.csv")
results_df = pd.read_csv(catalog_out)
# ----------------------------------------------

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# --- SELF-HEALING DATA CLEANING FOR PLOTTING ---
clean_sky_noise = results_df['Sky_RMS_Noise'].dropna()
clean_scatter_data = results_df.dropna(subset=['AB_Magnitude', 'Net_Aperture_Flux'])

# Panel A: Sky Noise Variance Histogram
if len(clean_sky_noise) > 0:
    ax1.hist(clean_sky_noise, bins=20, color='royalblue', edgecolor='black', alpha=0.85)
    # FIX: Using raw string (r"") and single backslashes ensures clean Matplotlib math parsing
    ax1.set_title(r"Vector Tracker: Background Sky Noise Variance ($\sigma_{\mathrm{sky}}$)", fontsize=11, fontweight='bold')
else:
    ax1.text(0.5, 0.5, "No Finite Sky Noise Data Found", ha='center', va='center')
    ax1.set_title("Vector Tracker: Background Sky Noise Variance", fontsize=11, fontweight='bold')

ax1.set_xlabel("RMS Background Variance (Digital Numbers)", fontsize=10)
ax1.set_ylabel("Number of Target Positions", fontsize=10)
ax1.grid(True, linestyle=':', alpha=0.6)

# Panel B: Calibrated Photometric Distribution Sequence
if len(clean_scatter_data) > 0:
    ax2.scatter(clean_scatter_data['AB_Magnitude'], 
                clean_scatter_data['Net_Aperture_Flux'], 
                c=clean_scatter_data['AB_Magnitude'], 
                cmap='plasma_r', edgecolor='black', alpha=0.75, s=45)
    # FIX: Standardized title formatting
    ax2.set_title(r"Photometric Response: Net Flux vs. Derived AB Mag", fontsize=11, fontweight='bold')
    ax2.set_yscale('log')
else:
    ax2.text(0.5, 0.5, "No Finite Photometric Data Found", ha='center', va='center')
    ax2.set_title(r"Photometric Response: Net Flux vs. Derived AB Mag", fontsize=11, fontweight='bold')

# FIX: Cleaning math symbols in axis labels using raw strings
ax2.set_xlabel("Calibrated Brightness Sequence (AB Magnitude)", fontsize=10)
ax2.set_ylabel(r"Net Integrated Flux ($F_{\mathrm{net}}$, DN)", fontsize=10)
ax2.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.savefig(os.path.join(stage3_dir, "vector_pipeline_diagnostic_distribution.png"), dpi=200)
plt.show()

### The diagnostic metrics for *all 4 shock filter channels* simultaneously.
We will expand to render an optimized 4x2 grid layout (8 graphics total). The left column will display the **Sky Noise Variance histograms** for each channel, and the right column will show the corresponding **Calibrated Photometric response curves**, automatically dropping NaN elements to stay safe under Pandas 3.0 rules.

- **The 4x2 Axis Matrix Loop**: Automatically creates 8 subplots inside a single high-resolution figure object (`axes[row, col]`), structuring your visual verification checks side-by-side.
- **Safe Layout Packing (`bbox_inches='tight'`)**: Prevents axis titles, labels, or overlapping exponent formats from clipping off the edges of your image files when rendering the huge 4-filter layout grid.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- ROBUST WORKSPACE DATA RE-LOAD ---
stage3_dir = "./processed_stage3/"
catalog_out = os.path.join(stage3_dir, "ic348_combined_master_science_catalog.csv")
results_df = pd.read_csv(catalog_out)
# --------------------------------------

# Define our 4 targeted shock tracking channels explicitly
filter_keys = ['F164N', 'F323N', 'F466N', 'F470N']
colors = ['royalblue', 'crimson', 'forestgreen', 'darkorange']

# Initialize an explicit 4x2 subplots matrix grid (8 graphics total)
fig, axes = plt.subplots(4, 2, figsize=(15, 18), sharex=False)
fig.suptitle(r"JWST Vector Data Reduction: Multi-Filter Diagnostic Matrix Overview", 
             fontsize=14, fontweight='bold', y=0.99)

for idx, (f_key, color) in enumerate(zip(filter_keys, colors)):
    # Local loop axis pointers
    ax_hist = axes[idx, 0]
    ax_scat = axes[idx, 1]
    
    # Verify columns exist before slicing data arrays
    flux_col = f'Flux_{f_key}'
    mag_col = f'Mag_{f_key}'
    
    if flux_col not in results_df.columns or mag_col not in results_df.columns:
        # Fallback text notification if a channel was skipped in the processing loop
        ax_hist.text(0.5, 0.5, f"Missing Channel Data: {f_key}", ha='center', va='center')
        ax_scat.text(0.5, 0.5, f"Missing Channel Data: {f_key}", ha='center', va='center')
        continue
        
    # --- DATA RECOVERY DROPPING OUTLIER NAN ENTRIES ---
    # Look for matching column values down our clean index structures
    clean_pair = results_df.dropna(subset=[flux_col, mag_col])
    # --------------------------------------------------
    
    # 📊 LEFT GRAPHICS: Sky background statistics approximation equivalent 
    # Since our multi-script uses a uniform background model, we track flux ranges
    if len(clean_pair) > 0:
        ax_hist.hist(clean_pair[mag_col], bins=20, color=color, edgecolor='black', alpha=0.8)
        ax_hist.set_title(f"Channel [{f_key}]: Component Candidate Count vs. Brightness", 
                          fontsize=10, fontweight='bold')
    else:
        ax_hist.text(0.5, 0.5, "No Finite Data", ha='center', va='center')
        
    ax_hist.set_xlabel("Calibrated Brightness (AB Magnitude)", fontsize=9)
    ax_hist.set_ylabel("Number of Candidates", fontsize=9)
    ax_hist.grid(True, linestyle=':', alpha=0.5)
    
    # 📊 RIGHT GRAPHICS: Calibrated Photometric Response Tracks
    if len(clean_pair) > 0:
        scatter = ax_scat.scatter(clean_pair[mag_col], clean_pair[flux_col], 
                                  c=clean_pair[mag_col], cmap='plasma_r', 
                                  edgecolor='black', alpha=0.75, s=40)
        ax_scat.set_title(f"Channel [{f_key}]: Photometric Core Response Spectrum", 
                          fontsize=10, fontweight='bold')
        ax_scat.set_yscale('log')
    else:
        ax_scat.text(0.5, 0.5, "No Finite Data", ha='center', va='center')
        
    ax_scat.set_xlabel("Calibrated Brightness (AB Magnitude)", fontsize=9)
    ax_scat.set_ylabel(r"Net Integrated Flux ($F_{\mathrm{net}}$, DN)", fontsize=9)
    ax_scat.grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
# Save our comprehensive 8-panel visualization grid layout safely to your local workspace
plt.savefig(os.path.join(stage3_dir, "vector_pipeline_multi_filter_diagnostics.png"), dpi=200, bbox_inches='tight')
plt.show()